# 真实集成：SelfCheck BERTScore

真实 API 采样，再由 BERTScore 的实际模型计算每个原句的最佳候选 F1 相似度。

## 真实集成前提

安装：`python3 -m pip install -e ".[openai,selfcheck-bertscore]"`；需要 API key 和 BERTScore 模型路径或模型标识。

代码单元没有预执行。填写 API 凭据或本地模型绝对路径后，从上到下运行；不要将 API key 保存到 notebook。

## 参数说明

- `BERTSCORE_LANG: str = 'zh'`：BERTScore 文本语言。
- `BERTSCORE_MODEL: str`：本地路径或可下载模型标识。
- `rescale_with_baseline: bool = True`：是否进行基线重标定。
- `NUM_SAMPLES: int = 5`：每句样本数。

## 步骤 1：配置真实 API 与 BERTScore

In [ ]:
from pathlib import Path
import sys
repo_root = Path.cwd()
if not (repo_root / "src" / "hallucination").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

import os
from getpass import getpass
from openai import OpenAI
from hallucination.detection.backends.api import OpenAIResponsesBackend
OPENAI_MODEL = "gpt-4.1-mini"
TEMPERATURE = 0.8
api_key = os.environ.get("OPENAI_API_KEY") or getpass("OpenAI API key: ")
base_url = os.environ.get("OPENAI_BASE_URL")
client = OpenAI(api_key=api_key, **({"base_url": base_url} if base_url else {}))
backend = OpenAIResponsesBackend(OPENAI_MODEL, client=client, temperature=TEMPERATURE)

from hallucination.detection.backends.embeddings import BERTScoreSimilarityBackend
BERTSCORE_LANG = "zh"
BERTSCORE_MODEL = "/absolute/path/to/your/bertscore-model"
similarity = BERTScoreSimilarityBackend(
    lang=BERTSCORE_LANG,
    model_type=BERTSCORE_MODEL,
    rescale_with_baseline=True,
)

## 步骤 2：运行 scorer

In [ ]:
from hallucination.detection.blackbox.selfcheck_bertscore import SelfCheckBERTScoreScorer
from hallucination.detection.pipelines.short_answer import BlackBoxSamplingPipeline
result = BlackBoxSamplingPipeline(backend, sentence_scorers=[SelfCheckBERTScoreScorer(similarity)], num_samples=5).detect_prompt("请用两句话介绍蓝鲸的食性和分类。")
for unit in result.units:
    print(unit.text, unit.method_scores[0].risk, unit.method_scores[0].raw["matches"])

## 结果解读与限制

BERTScore 相近不代表蕴含或事实正确。模型、语言、baseline 设置和版本必须在实验报告中固定。